# ✋ Sinaliza — Treinamento no Google Colab

Este notebook permite treinar os modelos do Sinaliza usando GPU gratuita do Colab.

**Pré-requisitos:**
- Dataset V-Librasil já processado (landmarks .npz)
- Código do projeto no Google Drive ou clonado do GitHub

## 1. Setup

In [ ]:
# Verificar GPU disponível
!nvidia-smi

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clonar repositório (ou copiar do Drive)
# Opção A: GitHub
# !git clone https://github.com/seu-usuario/sinaliza.git /content/sinaliza

# Opção B: Google Drive
!cp -r '/content/drive/MyDrive/Sinaliza' /content/sinaliza

%cd /content/sinaliza

In [ ]:
# Instalar dependências
!pip install -q mediapipe pydantic-settings optuna

## 2. Preparar Dataset

Se ainda não processou os landmarks, execute as células abaixo.

In [ ]:
# Download do dataset (se necessário)
# Configure o Kaggle API primeiro:
# from google.colab import files
# files.upload()  # upload kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

# !python scripts/download_dataset.py --output data/raw

In [ ]:
# Extrair landmarks (demora ~1-2h)
# !python scripts/extract_landmarks.py --input data/raw/v-librasil --output data/landmarks --workers 2

# Validar
# !python scripts/validate_landmarks.py --input data/landmarks

# Build dataset
# !python scripts/build_dataset.py --input data/landmarks --output data/processed

In [ ]:
# OU: Copiar dataset já processado do Drive
!cp -r '/content/drive/MyDrive/Sinaliza/data/processed' data/processed

# Verificar
!ls data/processed/
!echo "---"
!ls data/processed/*.npz | wc -l
print("arquivos .npz encontrados")

## 3. Treinar Modelos

In [ ]:
# Treinar Bi-LSTM
!python scripts/train.py --config configs/bilstm.yaml

In [ ]:
# Treinar Transformer
!python scripts/train.py --config configs/transformer.yaml

In [ ]:
# Treinar TCN
!python scripts/train.py --config configs/tcn.yaml

## 4. Avaliar e Comparar

In [ ]:
# Comparar os 3 modelos
!python scripts/compare_models.py \
    --checkpoints artifacts/checkpoints/best_*.pt \
    --output artifacts/comparison

In [ ]:
# Visualizar resultados
from IPython.display import Image, display, Markdown
import glob

for img in sorted(glob.glob('artifacts/comparison/*.png')):
    display(Image(img))

report = glob.glob('artifacts/comparison/*.md')
if report:
    display(Markdown(open(report[0]).read()))

## 5. Exportar Melhor Modelo

In [ ]:
# Exportar o melhor modelo para produção
# Substitua pelo checkpoint com melhor acurácia
!python scripts/export_model.py \
    --checkpoint artifacts/checkpoints/best_bilstm.pt \
    --output artifacts/sinaliza_model.pt

In [ ]:
# Salvar artefatos no Drive
!cp -r artifacts/ '/content/drive/MyDrive/Sinaliza/artifacts/'
print("Artefatos salvos no Google Drive!")